# Mortgage Calculator

Enter an **annualized interest rate**, a **term** (in years) and a **loan amount**. The output is the fixed **monthly payment**.

For a loan of principal $P$, monthly rate $r = \frac{\text{annual rate}}{12}$ and $n$ monthly payments, the standard amortization formula is

$$
M = P \cdot \frac{r (1 + r)^n}{(1 + r)^n - 1}
$$

When $r = 0$ the payment is simply $M = P / n$. The formula is derived from scratch below.


## Derivation

### Assumptions

1. The lender quotes a **nominal annual rate** $i$ that is **compounded monthly**, so the interest rate per month is $r = i / 12$. This is the convention for fixed-rate mortgages in the United States [1, 3].
2. The borrower receives $P$ at time $0$ and makes $n$ **equal payments** $M$, one at the **end** of each month, for $n = 12 \times \text{(term in years)}$ months.
3. Each month, interest accrues on the outstanding balance, and then the payment is applied. Whatever is left over after interest reduces the balance. The loan is **fully amortizing**: the balance after the last payment is exactly zero.

### Step 1: the balance recursion

Let $B_k$ be the balance owed immediately after the $k$-th payment, with $B_0 = P$. During month $k$ the balance grows by a factor $(1 + r)$ and then $M$ is paid, so

$$
B_k = (1 + r)\,B_{k-1} - M, \qquad k = 1, 2, \dots, n.
$$

### Step 2: unroll the recursion

Apply the recursion repeatedly:

$$
\begin{aligned}
B_1 &= (1+r)P - M \\
B_2 &= (1+r)B_1 - M = (1+r)^2 P - M(1+r) - M \\
B_3 &= (1+r)B_2 - M = (1+r)^3 P - M(1+r)^2 - M(1+r) - M \\
&\;\;\vdots \\
B_k &= (1+r)^k P - M \sum_{j=0}^{k-1} (1+r)^j .
\end{aligned}
$$

(The general line follows by induction: if it holds for $k-1$, multiplying by $(1+r)$ and subtracting $M$ gives it for $k$.)

### Step 3: sum the geometric series

For $r \neq 0$ the finite geometric series has the closed form [4]

$$
\sum_{j=0}^{k-1} (1+r)^j = \frac{(1+r)^k - 1}{(1+r) - 1} = \frac{(1+r)^k - 1}{r},
$$

so the balance after $k$ payments is

$$
B_k = (1+r)^k P - M\,\frac{(1+r)^k - 1}{r}.
$$

### Step 4: impose full amortization

The loan is paid off when $B_n = 0$:

$$
(1+r)^n P = M\,\frac{(1+r)^n - 1}{r}
\quad\Longrightarrow\quad
\boxed{\,M = P \cdot \frac{r\,(1+r)^n}{(1+r)^n - 1}\,}
$$

which is the formula used in this notebook. For $r = 0$ the series in Step 3 is simply $k$, so $B_n = P - nM = 0$ gives $M = P/n$. (Equivalently, take the limit $r \to 0$ of the boxed formula.)

### Equivalent view: present value of an annuity

Dividing the boxed formula through by $(1+r)^n$ gives

$$
P = M \cdot \frac{1 - (1+r)^{-n}}{r} = M \cdot a_{\overline{n}|r},
$$

where $a_{\overline{n}|r}$ is the actuarial symbol for the present value of an annuity-immediate of $1$ per period for $n$ periods at rate $r$ [2, 3]. In words: the amount lent equals the present value, at the loan's rate, of all the payments. This is the "actuarial method" that U.S. Truth in Lending rules use to define the relationship between the amount financed, the payments and the rate [5].

### Sources

1. Wikipedia contributors, "Mortgage calculator," *Wikipedia, The Free Encyclopedia*, section "Monthly payment formula." https://en.wikipedia.org/wiki/Mortgage_calculator
2. Stephen G. Kellison, *The Theory of Interest*, 3rd ed., McGraw-Hill/Irwin, 2009. Chapter 3 (basic annuities) and Chapter 5 (amortization schedules and sinking funds).
3. Samuel A. Broverman, *Mathematics of Investment and Credit*, 7th ed., ACTEX, 2017. Chapter 2 (valuation of annuities) and Chapter 3 (loan repayment).
4. Wikipedia contributors, "Geometric series," *Wikipedia, The Free Encyclopedia*, section on the finite sum. https://en.wikipedia.org/wiki/Geometric_series
5. Consumer Financial Protection Bureau, Regulation Z (12 CFR Part 1026), Appendix J, "Annual Percentage Rate Computations for Closed-End Credit Transactions," which defines the actuarial method. https://www.consumerfinance.gov/rules-policy/regulations/1026/j/

**Caveat.** Other jurisdictions quote rates differently. For example, Canadian fixed-rate mortgages are quoted with semi-annual compounding by law, so the monthly rate is $r = (1 + i/2)^{1/6} - 1$ rather than $i/12$. The formula in Step 4 still applies once the correct per-period rate $r$ is used.


In [6]:
def monthly_payment(annual_rate_pct, term_years, amount):
    """Fixed monthly payment for a fully amortizing loan.

    annual_rate_pct : annual interest rate in percent, e.g. 6.5 for 6.5%
    term_years      : loan term in years
    amount          : loan principal
    """
    n = int(round(term_years * 12))
    if n <= 0:
        raise ValueError("term must be positive")
    r = annual_rate_pct / 100.0 / 12.0
    if r == 0:
        return amount / n
    factor = (1 + r) ** n
    return amount * r * factor / (factor - 1)

## Enter your inputs here

In [7]:
annual_rate = 6.5      # annual interest rate in percent
term_years  = 30       # term in years
amount      = 400_000  # loan amount

payment = monthly_payment(annual_rate, term_years, amount)
total_paid = payment * term_years * 12

print(f"Monthly payment : {payment:,.2f}")
print(f"Total paid      : {total_paid:,.2f}")
print(f"Total interest  : {total_paid - amount:,.2f}")

Monthly payment : 2,528.27
Total paid      : 910,177.95
Total interest  : 510,177.95


## Interactive version

Move the sliders (or type into the boxes) and the payment updates automatically.

In [10]:
import ipywidgets as widgets
from IPython.display import display

rate_w   = widgets.FloatSlider(value=5.1, min=0.0, max=15.0, step=0.05,
                               description="Rate (%)", readout_format=".2f",
                               continuous_update=False)
term_w   = widgets.IntSlider(value=30, min=1, max=40, step=1, description="Term (yrs)")
amount_w = widgets.FloatText(value=650_000, step=10_000, description="Amount")
out      = widgets.HTML()

def update(_=None):
    p = monthly_payment(rate_w.value, term_w.value, amount_w.value)
    total = p * term_w.value * 12
    out.value = (f"<h3>Monthly payment: {p:,.2f}</h3>"
                 f"Total paid over {term_w.value} years: {total:,.2f}<br>"
                 f"Total interest: {total - amount_w.value:,.2f}")

for w in (rate_w, term_w, amount_w):
    w.observe(update, names="value")
update()

display(widgets.VBox([rate_w, term_w, amount_w, out]))

## Sanity check

A quick check that the formula reproduces a well known figure: 200,000 at 6% for 30 years should be about 1,199.10 per month.

In [9]:
assert abs(monthly_payment(6.0, 30, 200_000) - 1199.10) < 0.01
assert abs(monthly_payment(0.0, 10, 12_000) - 100.0) < 1e-9
print("ok")

ok


## Verify the derivation numerically

Run the balance recursion from Step 1 of the derivation, $B_k = (1+r)B_{k-1} - M$, with the payment from the closed-form formula. If the derivation is right, the balance after the last payment is zero (up to floating point error), and the closed form for $B_k$ from Step 3 matches the recursion at every month.

In [5]:
def amortization_schedule(annual_rate_pct, term_years, amount):
    """Month-by-month (balance, interest, principal) using the recursion B_k = (1+r) B_{k-1} - M."""
    n = int(round(term_years * 12))
    r = annual_rate_pct / 100.0 / 12.0
    M = monthly_payment(annual_rate_pct, term_years, amount)
    rows, balance = [], amount
    for k in range(1, n + 1):
        interest = r * balance
        principal = M - interest
        balance = balance + interest - M   # same as (1+r)*balance - M
        rows.append((k, interest, principal, balance))
    return rows

def balance_closed_form(annual_rate_pct, term_years, amount, k):
    """B_k from Step 3 of the derivation."""
    r = annual_rate_pct / 100.0 / 12.0
    M = monthly_payment(annual_rate_pct, term_years, amount)
    g = (1 + r) ** k
    return g * amount - M * (g - 1) / r

schedule = amortization_schedule(annual_rate, term_years, amount)

# 1. Final balance is zero
final_balance = schedule[-1][3]
assert abs(final_balance) < 1e-6, final_balance

# 2. Recursion agrees with the closed form for B_k at every month
max_err = max(abs(bal - balance_closed_form(annual_rate, term_years, amount, k))
              for k, _, _, bal in schedule)
assert max_err < 1e-6, max_err

print(f"Balance after final payment : {final_balance:.2e}")
print(f"Max |recursion - closed form|: {max_err:.2e}")
print()
print(f"{'month':>5} {'interest':>12} {'principal':>12} {'balance':>14}")
for k, i, p, b in schedule[:3] + schedule[-3:]:
    print(f"{k:>5} {i:>12,.2f} {p:>12,.2f} {b:>14,.2f}")

Balance after final payment : 7.87e-09
Max |recursion - closed form|: 7.62e-09

month     interest    principal        balance
    1     2,166.67       361.61     399,638.39
    2     2,164.71       363.56     399,274.83
    3     2,162.74       365.53     398,909.30
  358        40.64     2,487.63       5,015.75
  359        27.17     2,501.10       2,514.65
  360        13.62     2,514.65           0.00
